# Handling Ambiguity and Improving Clarity in Prompt Engineering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/ambiguity-clarity.ipynb)

## Overview

This tutorial focuses on two critical aspects of prompt engineering: identifying and resolving ambiguous prompts, and techniques for writing clearer prompts. These skills are essential for effective communication with AI models and obtaining more accurate and relevant responses.

## Motivation

Ambiguity in prompts can lead to inconsistent or irrelevant AI responses, while lack of clarity can result in misunderstandings and inaccurate outputs. By mastering these aspects of prompt engineering, you can significantly improve the quality and reliability of AI-generated content across various applications.

## Key Components

1. Why ambiguity causes variable outputs (entropy & probability distributions)
2. Identifying ambiguous prompts
3. Strategies for resolving ambiguity
4. Tokenization-induced ambiguity
5. Techniques for writing clearer prompts & the Clarity Checklist
6. Structured prompts for clarity
7. Consistency-based ambiguity measurement (with quantitative metrics)
8. Ambiguity as a feature (creative uses)
9. Practical examples and exercises

## Method Details

We'll use an open-source LLM (Qwen3-8B) running locally via HuggingFace Transformers to demonstrate various techniques for handling ambiguity and improving clarity in prompts. The tutorial will cover:

1. Setting up the environment with HuggingFace Transformers on Google Colab
2. Analyzing ambiguous prompts and their potential interpretations
3. Implementing strategies to resolve ambiguity, such as providing context and specifying parameters
4. Exploring techniques for writing clearer prompts, including using specific language and structured formats
5. Measuring prompt ambiguity via response consistency
6. Practical exercises to apply these concepts in real-world scenarios

## Conclusion

By the end of this tutorial, you'll have a solid understanding of how to identify and resolve ambiguity in prompts, as well as techniques for crafting clearer prompts. These skills will enable you to communicate more effectively with AI models, resulting in more accurate and relevant outputs across various applications.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Overview**
- Understand **Motivation**
- Understand **Key Components**
- Understand **Method Details**
- Understand **Conclusion**


## Setup

First, let's install the necessary libraries and load the open-source model locally on Colab.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Why Ambiguity Causes Variable Outputs

### The Probability Distribution Perspective

When a language model processes a prompt, it produces a **probability distribution** over its entire vocabulary for the next token. The shape of this distribution is the key to understanding why ambiguous prompts cause unpredictable outputs:

- **Ambiguous prompts** create **flat (high-entropy) distributions**. Many tokens have similar probabilities because the model is uncertain about the user's intent. "Tell me about it" — what is "it"? The model assigns non-trivial probability to tokens related to technology, relationships, movies, weather, and countless other topics. Sampling from this flat distribution is like rolling a many-sided die: the outcome varies wildly between runs.

- **Clear prompts** create **peaked (low-entropy) distributions**. The model is confident about what comes next because the prompt leaves little room for interpretation. "Explain how photosynthesis converts sunlight into chemical energy" concentrates probability mass on biology-related tokens. Sampling from a peaked distribution consistently produces similar outputs.

**Entropy** quantifies this: \$H = -\sum p_i \log p_i\$. Higher entropy means more uncertainty, more randomness, and less predictable generation. The cell below demonstrates this directly by examining the model's actual next-token distributions.

In [ ]:
import torch
import torch.nn.functional as F

def compute_next_token_entropy(prompt_text):
    """Compute entropy of the next-token probability distribution.

    Returns the entropy (in nats), the full probability vector,
    and the top-k tokens with their probabilities.
    """
    messages = [{"role": "user", "content": prompt_text}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Logits for the LAST position = prediction for the next token
    last_logits = outputs.logits[0, -1, :]
    probs = torch.softmax(last_logits, dim=-1)

    # Entropy: H = -sum(p * log(p)), skip zero-prob tokens
    log_probs = torch.log(probs + 1e-10)
    entropy = -(probs * log_probs).sum().item()

    return entropy, probs

# --- Compare ambiguous vs. clear prompts ---
prompts = [
    ("Ambiguous", "Tell me about it"),
    ("Clear",     "Explain how photosynthesis converts sunlight into chemical energy"),
]

print("Next-Token Entropy Comparison")
print("=" * 65)

for label, prompt in prompts:
    entropy, probs = compute_next_token_entropy(prompt)
    top_k = torch.topk(probs, 10)

    print(f"\n[{label}] \"{prompt}\"")
    print(f"  Entropy: {entropy:.4f} nats")
    print(f"  Top-10 tokens and probabilities:")
    for i in range(10):
        tok = tokenizer.decode(top_k.indices[i])
        p = top_k.values[i].item()
        print(f"    {tok!r:>15s}: {p:.4f}  ({p*100:.1f}%)")
    top10_mass = top_k.values.sum().item()
    print(f"  Top-10 probability mass: {top10_mass:.4f} ({top10_mass*100:.1f}%)")

print("\n" + "=" * 65)
print("Higher entropy  = flatter distribution  = unpredictable generation")
print("Lower entropy   = peaked distribution   = predictable generation")

### Mechanistic Explanation

The entropy numbers above reveal the **root cause** of inconsistent AI responses: it is not randomness in the model itself but rather the model faithfully reflecting the ambiguity present in the input.

When entropy is high, the model is effectively saying: *"I could go in many different directions here because you haven't told me which one you want."* Each time you sample, you may get a completely different topic, tone, or structure — not because the model is broken, but because many continuations are approximately equally valid.

When entropy is low, the model has a strong opinion about what should come next. Even with sampling enabled, the outputs cluster tightly around the same semantic content.

**This is the core insight**: improving prompt clarity is not about tricking the model — it is about giving it enough information to concentrate its probability mass where you want it.

## Identifying Ambiguous Prompts

Let's start by examining some ambiguous prompts and analyzing their potential interpretations.

In [ ]:
ambiguous_prompts = [
    "Tell me about the bank.",
    "What's the best way to get to school?",
    "Can you explain the theory?"
]

for prompt in ambiguous_prompts:
    analysis_prompt = f"Analyze the following prompt for ambiguity: '{prompt}'. Explain why it's ambiguous and list possible interpretations."
    messages = [{"role": "user", "content": analysis_prompt}]
    print(f"Prompt: {prompt}")
    print(generate(messages))
    print("-" * 50)

## Resolving Ambiguity

Now, let's explore strategies for resolving ambiguity in prompts.

In [ ]:
def resolve_ambiguity(prompt, context):
    """
    Resolve ambiguity in a prompt by providing additional context.

    Args:
    prompt (str): The original ambiguous prompt
    context (str): Additional context to resolve ambiguity

    Returns:
    str: The model's response to the clarified prompt
    """
    clarified_prompt = f"{context}\n\nBased on this context, {prompt}"
    messages = [{"role": "user", "content": clarified_prompt}]
    return generate(messages)

# Example usage
ambiguous_prompt = "Tell me about the bank."
contexts = [
    "You are a financial advisor discussing savings accounts.",
    "You are a geographer describing river formations."
]

for context in contexts:
    print(f"Context: {context}")
    print(f"Clarified response: {resolve_ambiguity(ambiguous_prompt, context)}")
    print("-" * 50)

## Tokenization-Induced Ambiguity

### How Tokenization Creates Hidden Ambiguity

Before the model sees your prompt, it is broken into **tokens** — subword units from a learned vocabulary. This tokenization step can itself introduce ambiguity:

- **Leading spaces matter**: ` Python` (with a space) and `Python` (without) may map to different token IDs, which feed into different embedding vectors.
- **Capitalization changes tokens**: `"the apple"` and `"The Apple"` tokenize differently, potentially activating different semantic pathways.
- **Context affects subword splits**: The same characters may be split into different subword combinations depending on what surrounds them.
- **Punctuation primes structure**: Ending a prompt with `"1."` vs `"-"` vs nothing creates different token sequences that prime the model for different output formats.

Understanding tokenization helps you see that what *looks* like the same prompt to a human can be a subtly **different input** to the model.

In [ ]:
# Examine how the same words tokenize differently in different contexts

print("Context-Dependent Tokenization")
print("=" * 60)

word_contexts = [
    ("bank", [
        "The bank approved the loan application.",
        "We fished along the river bank at sunset.",
    ]),
    ("lead", [
        "She will lead the team to victory.",
        "The old pipe was made of lead.",
    ]),
    ("fine", [
        "The weather is fine today.",
        "He received a hefty fine for speeding.",
    ]),
]

for word, sentences in word_contexts:
    print(f"\nWord: '{word}'")
    for sent in sentences:
        tokens = tokenizer.tokenize(sent)
        ids = tokenizer.encode(sent, add_special_tokens=False)
        print(f"  Context: \"{sent}\"")
        print(f"  Tokens:  {tokens}")
        print(f"  IDs:     {ids}")
    print()

# Leading-space sensitivity
print("Leading-Space Sensitivity")
print("-" * 40)
for text in ["Python", " Python", "python", " python"]:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text, add_special_tokens=False)
    print(f"  {text!r:>10s} -> tokens: {tokens},  ids: {ids}")

# Capitalization effects
print("\nCapitalization Effects")
print("-" * 40)
for text in ["The apple", "the apple", "THE APPLE", "the Apple"]:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text, add_special_tokens=False)
    print(f"  {text!r:>15s} -> tokens: {tokens},  ids: {ids}")

In [ ]:
# Punctuation and formatting create different token sequences,
# which prime the model toward different output structures.

format_variants = [
    ("Numbered list",  "List three benefits of exercise:\n1."),
    ("Dash list",      "List three benefits of exercise:\n-"),
    ("Bare colon",     "List three benefits of exercise:"),
    ("Inline request", "What are three benefits of exercise, briefly?"),
]

print("How Formatting Guides Model Behavior")
print("=" * 65)

for label, prompt in format_variants:
    tokens = tokenizer.tokenize(prompt)
    messages = [{"role": "user", "content": prompt}]
    response = generate(messages, max_new_tokens=150)
    print(f"\n[{label}]")
    print(f"  Prompt:  \"{prompt}\"")
    print(f"  Token count: {len(tokens)}")
    print(f"  Last 5 tokens: {tokens[-5:]}")
    print(f"  Response (first 200 chars):")
    print(f"    {response[:200]}")
    print("-" * 65)

print("\nKey insight: trailing punctuation/formatting tokens prime the model's")
print("next-token distribution. '1.' strongly predicts a numbered list,")
print("while a bare ':' leaves the format open — another source of ambiguity.")

## Techniques for Writing Clearer Prompts

Let's explore some techniques for writing clearer prompts to improve AI responses.

In [ ]:
def compare_prompt_clarity(original_prompt, improved_prompt):
    """
    Compare the responses to an original prompt and an improved, clearer version.

    Args:
    original_prompt (str): The original, potentially unclear prompt
    improved_prompt (str): An improved, clearer version of the prompt

    Returns:
    tuple: Responses to the original and improved prompts
    """
    original_messages = [{"role": "user", "content": original_prompt}]
    improved_messages = [{"role": "user", "content": improved_prompt}]
    original_response = generate(original_messages)
    improved_response = generate(improved_messages)
    return original_response, improved_response

# Example usage
original_prompt = "How do I make it?"
improved_prompt = "Provide a step-by-step guide for making a classic margherita pizza, including ingredients and cooking instructions."

original_response, improved_response = compare_prompt_clarity(original_prompt, improved_prompt)

print("Original Prompt Response:")
print(original_response)
print("\nImproved Prompt Response:")
print(improved_response)

In [ ]:
# The Clarity Checklist: Systematically reduce ambiguity in 5 steps
# Start with a maximally ambiguous prompt and improve it one dimension at a time.

improvements = [
    ("Original (ambiguous)",
     "Tell me about that thing."),

    ("Step 1 — Specify the SUBJECT (who/what)",
     "Tell me about the Python programming language."),

    ("Step 2 — Specify the ACTION (what to do)",
     "Explain the key features of the Python programming language."),

    ("Step 3 — Specify the FORMAT (how to output)",
     "Explain the key features of the Python programming language "
     "as a bulleted list."),

    ("Step 4 — Specify CONSTRAINTS (limits, style)",
     "Explain the key features of the Python programming language "
     "as a bulleted list. Cover exactly 5 features. "
     "Keep each bullet to one sentence."),

    ("Step 5 — Specify CONTEXT (background info)",
     "I am a Java developer learning Python. "
     "Explain the key features of the Python programming language "
     "as a bulleted list. Cover exactly 5 features that would be "
     "most surprising or different coming from Java. "
     "Keep each bullet to one sentence."),
]

print("The Clarity Checklist — Systematic Prompt Improvement")
print("=" * 70)

for step_label, prompt in improvements:
    messages = [{"role": "user", "content": prompt}]
    response = generate(messages, max_new_tokens=300, temperature=0.7)
    print(f"\n{'=' * 70}")
    print(f">> {step_label}")
    print(f"   Prompt: \"{prompt}\"")
    print(f"\n   Response:\n{response[:500]}")
    print()

### How the Clarity Checklist Works

Each step in the checklist **reduces the output space** by constraining one dimension of the response:

| Step | What it constrains | Effect on probability distribution |
|------|-------------------|-----------------------------------|
| **Subject** | Eliminates topic ambiguity | Concentrates mass on one domain |
| **Action** | Eliminates task ambiguity | Narrows the type of content (explain vs. list vs. compare) |
| **Format** | Eliminates structural ambiguity | Constrains token sequences to match the format |
| **Constraints** | Eliminates scope ambiguity | Limits length, count, style — further sharpening the distribution |
| **Context** | Eliminates audience/perspective ambiguity | Biases word choice and depth toward the target reader |

Think of it geometrically: each specification removes a **degree of freedom** from the response space. The original prompt "Tell me about that thing" has near-infinite degrees of freedom. The final prompt has very few — which is exactly what makes it produce consistent, useful output.

**Use this checklist** whenever you find yourself getting unpredictable results. Ask: *Which of these five dimensions have I left unspecified?*

## Structured Prompts for Clarity

Using structured prompts can significantly improve clarity and consistency in AI responses.

In [ ]:
def structured_analysis(topic, aspects, tone):
    """Generate a structured analysis using an f-string prompt."""
    prompt = (
        f"Provide an analysis of {topic} considering the following aspects:\n"
        f"1. {aspects[0]}\n"
        f"2. {aspects[1]}\n"
        f"3. {aspects[2]}\n\n"
        f"Present the analysis in a {tone} tone."
    )
    messages = [{"role": "user", "content": prompt}]
    return generate(messages)

# Example usage
response = structured_analysis(
    topic="the impact of social media on society",
    aspects=["communication patterns", "mental health", "information spread"],
    tone="balanced and objective",
)
print(response)

## Consistency-Based Ambiguity Measurement

A practical way to detect prompt ambiguity is to sample multiple responses at a higher temperature. If the model produces highly varied answers, the prompt is likely ambiguous. This technique leverages stochastic decoding to surface different interpretations the model assigns to the same input.

In [ ]:
def measure_ambiguity(prompt, num_samples=3):
    """Measure prompt ambiguity by checking response consistency."""
    messages = [{"role": "user", "content": prompt}]
    responses = [generate(messages, temperature=0.8, do_sample=True) for _ in range(num_samples)]
    # High variance = ambiguous prompt
    unique = len(set(r.strip()[:100] for r in responses))
    print(f"Unique responses: {unique}/{num_samples} (higher = more ambiguous)")
    return responses

# Compare an ambiguous prompt vs. a clear one
print("=== Ambiguous prompt ===")
ambiguous_responses = measure_ambiguity("Tell me about the bank.")
for i, r in enumerate(ambiguous_responses, 1):
    print(f"--- Response {i} (first 200 chars) ---")
    print(r[:200])
    print()

print("\n=== Clear prompt ===")
clear_responses = measure_ambiguity(
    "Explain how commercial banks generate revenue through interest rate spreads, "
    "focusing on the difference between deposit rates and lending rates."
)
for i, r in enumerate(clear_responses, 1):
    print(f"--- Response {i} (first 200 chars) ---")
    print(r[:200])
    print()

In [ ]:
import re
from collections import Counter

def extract_keywords(text, top_n=15):
    """Extract top-N keywords by frequency, excluding common stop words."""
    stop_words = {
        'the','a','an','is','are','was','were','be','been','being','have','has',
        'had','do','does','did','will','would','could','should','may','might',
        'shall','can','to','of','in','for','on','with','at','by','from','as',
        'into','through','during','it','its','this','that','these','those',
        'and','but','or','not','no','if','then','than','so','up','out','about',
        'also','more','most','very','just','how','what','which','who','where',
        'when','why','all','each','every','some','any','many','much','such',
        'own','other','another','new','old','first','last','one','two','three',
    }
    words = re.findall(r'\b[a-z]{3,}\b', text.lower())
    words = [w for w in words if w not in stop_words]
    return set(dict(Counter(words).most_common(top_n)).keys())

def measure_generation_consistency(prompt, num_samples=5):
    """Generate multiple responses and measure their consistency.

    Returns keyword overlap (Jaccard), length coefficient of variation,
    and the raw responses.
    """
    messages = [{"role": "user", "content": prompt}]
    responses = [
        generate(messages, temperature=0.8, do_sample=True, max_new_tokens=256)
        for _ in range(num_samples)
    ]

    # 1. Keyword overlap — pairwise Jaccard similarity
    kw_sets = [extract_keywords(r) for r in responses]
    overlaps = []
    for i in range(len(kw_sets)):
        for j in range(i + 1, len(kw_sets)):
            inter = kw_sets[i] & kw_sets[j]
            union = kw_sets[i] | kw_sets[j]
            overlaps.append(len(inter) / len(union) if union else 0)
    avg_overlap = sum(overlaps) / len(overlaps) if overlaps else 0

    # 2. Response length variance (coefficient of variation)
    lengths = [len(r.split()) for r in responses]
    mean_len = sum(lengths) / len(lengths)
    variance = sum((l - mean_len) ** 2 for l in lengths) / len(lengths)
    cv = variance ** 0.5 / mean_len if mean_len > 0 else 0

    return {
        "avg_keyword_overlap": avg_overlap,
        "length_cv": cv,
        "lengths": lengths,
        "responses": responses,
    }

# --- Test on a spectrum of prompts ---
test_prompts = [
    ("Ambiguous",  "Tell me about the bank."),
    ("Ambiguous",  "What should I do?"),
    ("Clear",      "Explain the three main functions of commercial banks: "
                   "accepting deposits, making loans, and facilitating payments."),
    ("Clear",      "List 5 evidence-based strategies for improving sleep quality, "
                   "with a brief explanation of each."),
]

print("Generation Consistency Analysis  (5 samples each)")
print("=" * 70)
for label, prompt in test_prompts:
    r = measure_generation_consistency(prompt)
    ellipsis = '...' if len(prompt) > 70 else ''
    print(f"\n[{label}] \"{prompt[:70]}{ellipsis}\"")
    print(f"  Keyword overlap (Jaccard avg): {r['avg_keyword_overlap']:.3f}  "
          f"(1.0 = identical topics)")
    print(f"  Length coeff. of variation:     {r['length_cv']:.3f}  "
          f"(0.0 = identical lengths)")
    print(f"  Response lengths (words):       {r['lengths']}")
    print(f"  Sample (first 150 chars):       {r['responses'][0][:150]}...")
    print("-" * 70)

### Interpreting the Consistency Metrics

The two metrics above give you a **concrete, quantifiable measure** of prompt ambiguity:

| Metric | High Value | Low Value |
|--------|-----------|-----------|
| **Keyword overlap** (Jaccard) | Responses discuss the same topic — low ambiguity | Responses diverge into different topics — high ambiguity |
| **Length CV** (coefficient of variation) | Response lengths vary wildly — the model is uncertain about scope | Consistent lengths — the model agrees on the expected depth |

**Practical rule of thumb**: If keyword overlap < 0.3 or length CV > 0.5, your prompt is likely ambiguous and needs refinement. These metrics can be automated into a prompt quality checker during development.

In [ ]:
# Sometimes ambiguity is a TOOL, not a bug.
# Deliberately vague prompts + high temperature = diverse, creative outputs.

creative_prompt = "Write a short opening line for a story."

print("Ambiguity as a Creative Tool")
print("=" * 60)

# Low temperature — outputs converge
print("\n--- Low temperature (0.3) — Convergent ---")
for i in range(4):
    messages = [{"role": "user", "content": creative_prompt}]
    response = generate(messages, max_new_tokens=60, temperature=0.3)
    first_line = response.strip().split("\n")[0]
    print(f"  {i+1}. {first_line}")

# High temperature — outputs diverge
print("\n--- High temperature (1.2) — Divergent ---")
for i in range(4):
    messages = [{"role": "user", "content": creative_prompt}]
    response = generate(messages, max_new_tokens=60, temperature=1.2)
    first_line = response.strip().split("\n")[0]
    print(f"  {i+1}. {first_line}")

# Specific prompt at high temperature — creative WITHIN constraints
specific = ("Write a short opening line for a gothic horror story "
            "set in a Victorian mansion.")
print(f"\n--- Specific prompt at high temperature (1.2) ---")
print(f'    Prompt: "{specific}"')
for i in range(4):
    messages = [{"role": "user", "content": specific}]
    response = generate(messages, max_new_tokens=60, temperature=1.2)
    first_line = response.strip().split("\n")[0]
    print(f"  {i+1}. {first_line}")

print("\nVague prompt + high temp  = wildly different genres and styles")
print("Specific prompt + high temp = creative WITHIN specified constraints")
print("The interaction between prompt specificity and temperature is key.")

### Ambiguity Is Not Always Bad — It Is a Tool

The examples above illustrate a crucial nuance:

- **Accidental ambiguity** produces frustrating, unusable variation. You wanted a specific answer but got randomness.
- **Intentional ambiguity** produces useful, creative diversity. You *wanted* the model to explore freely.

The difference is **intent**. When brainstorming, exploring, or generating creative content, a vague prompt combined with a high temperature setting is exactly right — it gives the model permission to surprise you. When you need a precise, reliable answer, the Clarity Checklist from the previous section is your friend.

**Master both modes**: the ability to move fluidly between tight specification and open exploration is one of the hallmarks of expert prompt engineering.

## Practical Exercise: Improving Prompt Clarity

Now, let's practice improving the clarity of prompts.

In [ ]:
unclear_prompts = [
    "What's the difference?",
    "How does it work?",
    "Why is it important?"
]

def improve_prompt_clarity(unclear_prompt):
    """
    Improve the clarity of a given prompt.

    Args:
    unclear_prompt (str): The original unclear prompt

    Returns:
    str: An improved, clearer version of the prompt
    """
    improvement_prompt = f"The following prompt is unclear: '{unclear_prompt}'. Please provide a clearer, more specific version of this prompt. Output just the improved prompt and nothing else."
    messages = [{"role": "user", "content": improvement_prompt}]
    return generate(messages)

for prompt in unclear_prompts:
    improved_prompt = improve_prompt_clarity(prompt)
    print(f"Original: {prompt}")
    print(f"Improved: {improved_prompt}")
    print("-" * 50)

## 📝 Key Takeaways

- **Overview** — revisit this section for reference
- **Motivation** — revisit this section for reference
- **Key Components** — revisit this section for reference
- **Method Details** — revisit this section for reference
- **Conclusion** — revisit this section for reference


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the prompt-engineering/ directory

---

## 🧭 Navigation

➡️ **Next:** [Basic-Prompt-Structures](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/basic-prompt-structures.ipynb)